# Эксперименты

Цель - сравнить способы ранжирования регламентов и выбрать
воспроизводимую конфигурацию для `run.py`

Официальной разметки нет. Для сравнения моделей используется небольшая
ручная выборка, поэтому локальные метрики служат только ориентиром

План экспериментов:

- E0 - word TF-IDF с униграммами
- E1 - word TF-IDF с униграммами и биграммами
- E2 - char TF-IDF
- E2b - BM25
- E3 - комбинация word и char TF-IDF
- E4a-E4c - абляция дополнительного контекста регламентов
- E5 - мультиязычная модель эмбеддингов
- E6 - объединение E2 и E5 с помощью Reciprocal Rank Fusion
- E7 - переранжирование top-30 E6 с помощью cross-encoder
- однократная проверка выбранной конфигурации на holdout

In [1]:
from pathlib import Path
from time import perf_counter
import re

import numpy as np
import pandas as pd

In [2]:
from tnved_ranker.data import load_data
from tnved_ranker.evaluation import calculate_ranking_metrics
from tnved_ranker.retrieval import rank_regulations
from tnved_ranker.validation import (
    PREDICTION_COLUMNS,
    validate_predictions,
)

## E0 - word TF-IDF baseline

**Запрос:** `G31_1 + desc_extention`

**Документ:** `description`

**Модель:** word TFIDF с настройками по умолчанию

**Анализатор:** word

**Сходство:** cosine similarity

**Назначение:** базовая точка сравнения для последующих экспериментов

In [3]:
declarations, regulations = load_data("data")

started_at = perf_counter()

predictions_baseline = rank_regulations(
    declarations,
    regulations,
    top_k=10,
)

elapsed = perf_counter() - started_at

validate_predictions(
    predictions_baseline,
    declarations,
    regulations,
    top_k=10,
)

print(f"Время: {elapsed:.3f} секунды")
print(f"Размер результата: {predictions_baseline.shape}")

Path("out").mkdir(exist_ok=True)

predictions_baseline.to_csv(
    "out/predictions_baseline.csv",
    index=False,
)

Время: 0.265 секунды
Размер результата: (1200, 4)


In [4]:
sample_ids = ["D0001", "D0002", "D0003"]

top3 = (
    predictions_baseline[
        predictions_baseline["declaration_id"].isin(sample_ids)
        & predictions_baseline["rank"].le(3)
    ]
    .merge(
        regulations[
            ["regulation_id", "code", "description"]
        ],
        on="regulation_id",
    )
)

display(top3)

,declaration_id,rank,regulation_id,score,code,description
0,D0001,1,R0003,0.469308,0203295909,"Мясо домашних свиней, замороженное, прочее"
1,D0001,2,R0002,0.289236,0203295901,"Мясо домашних свиней, замороженное, прочее, в ..."
2,D0001,3,R0001,0.271475,0203295502,"Мясо обваленное домашних свиней, замороженное,..."
3,D0002,1,R0036,0.216562,0802700000,"Орехи колы (cola spp.), свежие или сушеные, оч..."
4,D0002,2,R0025,0.211809,0402915100,"Молоко и сливки в прочих видах, без добавления..."
5,D0002,3,R0217,0.191744,7223001100,"Проволока из коррозионностойкой стали, содержа..."
6,D0003,1,R0107,0.272448,2922430000,Кислота антраниловая и ее соли
7,D0003,2,R0131,0.223593,3913100000,"Кислота альгиновая, ее соли и сложные эфиры"
8,D0003,3,R0104,0.203959,2917392000,Сложный эфир или ангидрид тетрабромфталевой ки...


### Результат E0

- Время выполнения: 0,18 секунды
- Матрица документов: 360 × 1530
- Матрица запросов: 120 × 1530
- Матрица сходства: 120 × 360
- Результат: 1200 строк
- Проверка формата: пройдена

У декларации `D0011` четыре последних кандидата имеют нулевой score.
Это означает, что word TF-IDF не нашел общих слов и не может содержательно
упорядочить нижнюю часть top-10

E0 используется как базовая точка сравнения

## Ручная проверочная выборка

Для приблизительного сравнения экспериментов используется ручная выборка:

- 10 случайных dev-примеров
- 5 сложных dev-примеров
- 5 случайных holdout-примеров

Dev используется для выбора архитектуры. Holdout проверяется один раз после
выбора финальной конфигурации

Ручные метки не загружаются рабочим пайплайном и не влияют на отдельные
декларации

In [5]:
SEED = 42

already_inspected = {
    "D0001",
    "D0002",
    "D0003",
}

available_ids = declarations.loc[
    ~declarations["declaration_id"].isin(already_inspected),
    "declaration_id",
]

In [6]:
holdout_ids = available_ids.sample(n=5, random_state=SEED).tolist()
holdout_ids

['D0048', 'D0008', 'D0057', 'D0046', 'D0014']

In [7]:
declaration_texts = (
    declarations
    .set_index("declaration_id")[
        ["G31_1", "desc_extention"]
    ]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.lower()
)

challenge_patterns = {
    "numbers": (
        r"\d+(?:[.,]\d+)?\s*"
        r"(?:%|кг|мм|см|см3|л|квт|лет|год)"
    ),
    "negation": (
        r"\b(?:не|без|отсутств\w*)\b"
    ),
    "materials": (
        r"\b(?:стал\w*|металл\w*|пластмасс\w*|"
        r"полимер\w*|текстил\w*|дерев\w*|кож\w*)\b"
    ),
    "condition": (
        r"\b(?:нов\w*|бывш\w*|б/у|употреблен\w*|"
        r"заморож\w*|сушен\w*|необработан\w*)\b"
    ),
}

In [8]:
used_ids = set(already_inspected) | set(holdout_ids)
challenge_ids = {}

for offset, (category, pattern) in enumerate(challenge_patterns.items(),start=1):
    candidates = declaration_texts[
        declaration_texts.str.contains(pattern, regex=True)
        & ~declaration_texts.index.isin(used_ids)
    ].index

    selected_id = (
        pd.Series(candidates)
        .sample(n=1, random_state=SEED + offset)
        .iloc[0]
    )

    challenge_ids[category] = selected_id
    used_ids.add(selected_id)

In [9]:
max_scores = predictions_baseline.groupby(
    "declaration_id"
)["score"].max()

low_score_id = (
    max_scores
    .drop(labels=list(used_ids), errors="ignore")
    .idxmin()
)

challenge_ids["low_lexical_score"] = low_score_id
used_ids.add(low_score_id)

challenge_ids

{'numbers': 'D0082',
 'negation': 'D0099',
 'materials': 'D0088',
 'condition': 'D0023',
 'low_lexical_score': 'D0040'}

In [10]:
remaining_ids = available_ids[
    ~available_ids.isin(
        holdout_ids + list(challenge_ids.values())
    )
]

random_dev_ids = remaining_ids.sample(n=10, random_state=SEED + 100).tolist()

dev_ids = random_dev_ids + list(challenge_ids.values())

assert len(dev_ids) == 15
assert len(holdout_ids) == 5
assert set(dev_ids).isdisjoint(holdout_ids)

In [11]:
manual_evaluation_path = Path("manual_evaluation.csv")

if manual_evaluation_path.exists():
    manual_evaluation = pd.read_csv(
        manual_evaluation_path,
        dtype={
            "declaration_id": "string",
            "expected_regulation_id": "string",
            "alternative_regulation_id": "string",
        },
    )
else:
    selection_type = {
        declaration_id: "random"
        for declaration_id in random_dev_ids + holdout_ids
    }
    selection_type.update({
        declaration_id: category
        for category, declaration_id in challenge_ids.items()
    })

    manual_evaluation = pd.DataFrame({
        "declaration_id": dev_ids + holdout_ids,
    })
    manual_evaluation["split"] = manual_evaluation[
        "declaration_id"
    ].apply(
        lambda value: "holdout" if value in holdout_ids else "dev"
    )
    manual_evaluation["selection_type"] = manual_evaluation[
        "declaration_id"
    ].map(selection_type)
    manual_evaluation["expected_regulation_id"] = pd.NA
    manual_evaluation["confidence"] = pd.NA
    manual_evaluation["alternative_regulation_id"] = pd.NA
    manual_evaluation["rationale"] = pd.NA

    manual_evaluation = manual_evaluation[
        [
            "split",
            "selection_type",
            "declaration_id",
            "expected_regulation_id",
            "confidence",
            "alternative_regulation_id",
            "rationale",
        ]
    ]
    manual_evaluation.to_csv(
        manual_evaluation_path,
        index=False,
    )

manual_evaluation.query("split == 'dev'")

,split,selection_type,declaration_id,expected_regulation_id,confidence,alternative_regulation_id,rationale
0,dev,random,D0032,R0019,high,<NA>,Совпадают сурими из рыбного фарша и мороженое ...
1,dev,random,D0056,R0172,high,R0173,Синтетическое неотбеленное трикотажное полотно...
2,dev,random,D0103,R0063,high,R0062,"Дата относится к периоду август-декабрь, а цен..."
3,dev,random,D0119,R0228,medium,R0226,Бурильная бесшовная труба из коррозионностойко...
4,dev,random,D0107,R0163,high,R0165,Полиэфирное волокно прочесано и подготовлено д...
5,dev,random,D0085,R0268,medium,R0269,Режущая головка является частью водоструйной м...
6,dev,random,D0012,R0133,high,R0135,Резиновая конвейерная лента армирована только ...
7,dev,random,D0079,R0193,high,R0195,Головной убор изготовлен из мехового фетра и с...
8,dev,random,D0043,R0308,high,R0309,Буксовый узел моторного вагона трамвая прямо с...
9,dev,random,D0064,R0179,high,R0178,Мужской трикотажный джемпер из химического акр...


### Протокол ручной разметки

1. Выборка деклараций создаётся до сравнения улучшенных моделей
2. Holdout выбирается случайно с фиксированным `SEED`
3. Для разметки используются `G31_1` и `desc_extention`
4. Кандидаты ищутся по `description`, `notes` и `explanation`
5. Для неоднозначных кодов проверяется справочник ТН ВЭД
6. Выбор выполняется независимо от текущего ранжирования модели
7. Предпочтение отдается наиболее конкретному регламенту, все условия
   которого согласуются с декларацией
8. Числа, диапазоны, единицы измерения, отрицания, материал и состояние
   товара проверяются явно
9. Для каждого примера сохраняются основной ответ, ближайшая альтернатива,
   уверенность и краткое обоснование
10. Метки `low` не используются для строгого сравнения моделей

Ручная разметка - локальный ориентир, а не официальная разметка задания

In [12]:
def show_declaration(declaration_id: str) -> None:
    row = declarations.loc[
        declarations["declaration_id"].eq(declaration_id),
        [
            "declaration_id",
            "G31_1",
            "desc_extention",
        ],
    ]

    display(row.T)

In [13]:
REGULATION_SEARCH_FIELDS = [
    "description",
    "notes",
    "explanation",
]

regulation_search_texts = (
    regulations[REGULATION_SEARCH_FIELDS]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.lower()
)


def find_regulations(term: str) -> pd.DataFrame:
    mask = regulation_search_texts.str.contains(
        term.lower(),
        regex=False,
    )

    return regulations.loc[
        mask,
        [
            "regulation_id",
            "code",
            "description",
            "notes",
            "explanation",
        ],
    ]

In [14]:
current_declaration_id = manual_evaluation.iloc[0].declaration_id
show_declaration(current_declaration_id)

,31
declaration_id,D0032
G31_1,"СУРИМИ ИЗ МИНТАЯ, МОРОЖЕНОЕ, В ВИДЕ ПРЕССОВАНН..."
desc_extention,"РЫБНЫЙ БЕЛКОВЫЙ ФАРШ, БЕЗ ДОБАВЛЕНИЯ ИНЫХ ВИДО..."


In [15]:
declarations[declarations["declaration_id"] == current_declaration_id]["desc_extention"].values[0]

'РЫБНЫЙ БЕЛКОВЫЙ ФАРШ, БЕЗ ДОБАВЛЕНИЯ ИНЫХ ВИДОВ РЫБЫ; НЕТТО 5000 КГ, ТЕМПЕРАТУРА -18 C'

In [16]:
find_regulations("сурими")

,regulation_id,code,description,notes,explanation
292,R0019,0304932000,"Прочее мясо рыбы (включая фарш) сурими, мороженое","ГРУППА 03 РЫБА И РАКООБРАЗНЫЕ, МОЛЛЮСКИ И ПРОЧ...","0304 93 – – тилапии ( Oreochromis spp .), сома..."


In [17]:
saved_dev_ids = set(
    manual_evaluation.loc[
        manual_evaluation["split"].eq("dev"),
        "declaration_id",
    ]
)

saved_holdout_ids = set(
    manual_evaluation.loc[
        manual_evaluation["split"].eq("holdout"),
        "declaration_id",
    ]
)

assert len(manual_evaluation) == len(dev_ids) + len(holdout_ids)
assert saved_dev_ids == set(dev_ids)
assert saved_holdout_ids == set(holdout_ids)

assert manual_evaluation["confidence"].isin(
    ["high", "medium", "low"]
).all()

assert manual_evaluation["expected_regulation_id"].isin(
    regulations["regulation_id"]
).all()

assert (
    manual_evaluation["rationale"]
    .fillna("")
    .str.strip()
    .ne("")
    .all()
)

display(
    pd.crosstab(
        manual_evaluation["split"],
        manual_evaluation["confidence"],
        margins=True,
    )
)

confidence,high,medium,All
split,,,
dev,13,2,15
holdout,4,1,5
All,17,3,20


### Результат ручной разметки

- Размечены 15 dev-примеров и 5 holdout-примеров. Для каждого примера
сохранены ожидаемый регламент, ближайшая альтернатива, уверенность
и краткое обоснование

- Holdout не используется при выборе конфигурации и проверяется один раз
после ее фиксации

## Локальная оценка E0

In [18]:
dev_labels = manual_evaluation.query(
    "split == 'dev' and confidence != 'low'"
)

e0_metrics = calculate_ranking_metrics(
    predictions_baseline,
    dev_labels,
)

pd.Series(e0_metrics)

n_queries    15.000000
hit_at_1      0.333333
hit_at_5      0.800000
hit_at_10     0.800000
mrr_at_10     0.487778
dtype: float64

In [19]:
assert e0_metrics["n_queries"] == len(dev_labels)

for name, value in e0_metrics.items():
    if name != "n_queries":
        assert 0 <= value <= 1

In [20]:
dev_ranks = (
    dev_labels
    .merge(
        predictions_baseline[
            [
                "declaration_id",
                "regulation_id",
                "rank",
                "score",
            ]
        ],
        left_on=[
            "declaration_id",
            "expected_regulation_id",
        ],
        right_on=[
            "declaration_id",
            "regulation_id",
        ],
        how="left",
    )
    .drop(columns="regulation_id")
    .rename(
        columns={
            "rank": "expected_rank",
            "score": "expected_score",
        }
    )
)

display(
    dev_ranks[
        [
            "declaration_id",
            "selection_type",
            "expected_regulation_id",
            "expected_rank",
            "expected_score",
            "confidence",
        ]
    ].sort_values(
        "expected_rank",
        na_position="last",
    )
)

,declaration_id,selection_type,expected_regulation_id,expected_rank,expected_score,confidence
0,D0032,random,R0019,1.0,0.497104,high
6,D0012,random,R0133,1.0,0.453827,high
7,D0079,random,R0193,1.0,0.356767,high
10,D0082,numbers,R0182,1.0,0.558353,high
13,D0023,condition,R0311,1.0,0.192272,high
1,D0056,random,R0172,2.0,0.357942,high
4,D0107,random,R0163,2.0,0.184602,high
3,D0119,random,R0228,3.0,0.248201,medium
12,D0088,materials,R0219,3.0,0.328983,high
8,D0043,random,R0308,4.0,0.149555,high


## E1 - word TF-IDF с биграммами

**Гипотеза:** биграммы помогут различать устойчивые товарные выражения

**Изменение:** `ngram_range=(1, 2)`

**Анализатор:** word

**Остальные компоненты:** без изменений

In [21]:
e1_predictions = rank_regulations(
    declarations,
    regulations,
    top_k=10,
    analyzer="word",
    ngram_range=(1, 2),
)

validate_predictions(
    e1_predictions,
    declarations,
    regulations,
    top_k=10,
)

e1_metrics = calculate_ranking_metrics(
    e1_predictions,
    dev_labels,
)

pd.Series(e1_metrics)

n_queries    15.000000
hit_at_1      0.333333
hit_at_5      0.800000
hit_at_10     0.800000
mrr_at_10     0.490000
dtype: float64

In [22]:
experiment_results = pd.DataFrame(
    [
        {
            "experiment": "E0",
            "configuration": "word TF-IDF, (1, 1)",
            **e0_metrics,
        },
        {
            "experiment": "E1",
            "configuration": "word TF-IDF, (1, 2)",
            **e1_metrics,
        },
    ]
)

experiment_results

,experiment,configuration,n_queries,hit_at_1,hit_at_5,hit_at_10,mrr_at_10
0,E0,"word TF-IDF, (1, 1)",15,0.333333,0.8,0.8,0.487778
1,E1,"word TF-IDF, (1, 2)",15,0.333333,0.8,0.8,0.490000


In [23]:
def get_expected_ranks(
    experiment_predictions: pd.DataFrame,
    labels: pd.DataFrame,
) -> pd.Series:
    comparison = experiment_predictions.merge(
        labels[
            [
                "declaration_id",
                "expected_regulation_id",
            ]
        ],
        on="declaration_id",
        how="inner",
    )

    relevant = comparison[
        comparison["regulation_id"].eq(
            comparison["expected_regulation_id"]
        )
    ]

    return relevant.groupby(
        "declaration_id"
    )["rank"].min()


e0_ranks = get_expected_ranks(
    predictions_baseline,
    dev_labels,
)

e1_ranks = get_expected_ranks(
    e1_predictions,
    dev_labels,
)

rank_comparison = dev_labels[
    [
        "declaration_id",
        "expected_regulation_id",
        "selection_type",
    ]
].copy()

rank_comparison["e0_rank"] = (
    rank_comparison["declaration_id"].map(e0_ranks)
)

rank_comparison["e1_rank"] = (
    rank_comparison["declaration_id"].map(e1_ranks)
)

rank_comparison["rank_change"] = (
    rank_comparison["e0_rank"]
    - rank_comparison["e1_rank"]
)

display(
    rank_comparison.sort_values(
        "rank_change",
        na_position="last",
        ascending=False,
    )
)

,declaration_id,expected_regulation_id,selection_type,e0_rank,e1_rank,rank_change
3,D0119,R0228,random,3.0,2.0,1.0
4,D0107,R0163,random,2.0,1.0,1.0
1,D0056,R0172,random,2.0,2.0,0.0
0,D0032,R0019,random,1.0,1.0,0.0
2,D0103,R0063,random,5.0,5.0,0.0
7,D0079,R0193,random,1.0,1.0,0.0
11,D0099,R0052,negation,5.0,5.0,0.0
10,D0082,R0182,numbers,1.0,1.0,0.0
13,D0023,R0311,condition,1.0,1.0,0.0
6,D0012,R0133,random,1.0,2.0,-1.0


### Результат E1

- Добавление биграмм не изменило `Hit@1`, `Hit@5` и `Hit@10`.
`MRR@10` вырос только с 0,488 до 0,490

- E0 остается более простым word-baseline

## E2 - символьный TF-IDF

**Гипотеза:** символьные n-граммы могут быть устойчивее к русским
словоформам, сокращениям и небольшим различиям в написании

**Изменение:** `ngram_range=(3, 5)`

**Анализатор:** char_wb

**Остальные компоненты:** без изменений

In [24]:
e2_predictions = rank_regulations(
    declarations,
    regulations,
    top_k=10,
    analyzer="char_wb",
    ngram_range=(3, 5),
)

validate_predictions(
    e2_predictions,
    declarations,
    regulations,
    top_k=10,
)

e2_metrics = calculate_ranking_metrics(
    e2_predictions,
    dev_labels,
)

pd.Series(e2_metrics)

n_queries    15.000000
hit_at_1      0.466667
hit_at_5      0.933333
hit_at_10     0.933333
mrr_at_10     0.666667
dtype: float64

In [25]:
experiment_results = pd.DataFrame(
    [
        {
            "experiment": "E0",
            "configuration": "word TF-IDF, (1, 1)",
            **e0_metrics,
        },
        {
            "experiment": "E1",
            "configuration": "word TF-IDF, (1, 2)",
            **e1_metrics,
        },
        {
            "experiment": "E2",
            "configuration": "char_wb TF-IDF, (3, 5)",
            **e2_metrics,
        }
    ]
)

experiment_results

,experiment,configuration,n_queries,hit_at_1,hit_at_5,hit_at_10,mrr_at_10
0,E0,"word TF-IDF, (1, 1)",15,0.333333,0.800000,0.800000,0.487778
1,E1,"word TF-IDF, (1, 2)",15,0.333333,0.800000,0.800000,0.490000
2,E2,"char_wb TF-IDF, (3, 5)",15,0.466667,0.933333,0.933333,0.666667


In [26]:
e2_ranks = get_expected_ranks(
    e2_predictions,
    dev_labels,
)

rank_comparison = dev_labels[
    [
        "declaration_id",
        "expected_regulation_id",
        "selection_type",
    ]
].copy()

rank_comparison["e0_rank"] = (
    rank_comparison["declaration_id"].map(e0_ranks)
)

rank_comparison["e2_ranks"] = (
    rank_comparison["declaration_id"].map(e2_ranks)
)

rank_comparison["rank_change"] = (
    rank_comparison["e0_rank"]
    - rank_comparison["e2_ranks"]
)

display(
    rank_comparison.sort_values(
        "rank_change",
        na_position="last",
        ascending=False,
    )
)

,declaration_id,expected_regulation_id,selection_type,e0_rank,e2_ranks,rank_change
11,D0099,R0052,negation,5.0,1.0,4.0
2,D0103,R0063,random,5.0,2.0,3.0
8,D0043,R0308,random,4.0,1.0,3.0
1,D0056,R0172,random,2.0,1.0,1.0
4,D0107,R0163,random,2.0,1.0,1.0
0,D0032,R0019,random,1.0,1.0,0.0
3,D0119,R0228,random,3.0,3.0,0.0
7,D0079,R0193,random,1.0,1.0,0.0
12,D0088,R0219,materials,3.0,3.0,0.0
6,D0012,R0133,random,1.0,2.0,-1.0


### Результат E2

- Char TF-IDF улучшил все локальные метрики относительно E0:
  `Hit@1` вырос с 0,333 до 0,467, `Hit@10` - с 0,800 до 0,933,
  а `MRR@10` - с 0,488 до 0,667

- Из трех регламентов, отсутствовавших в top-10 E0, два появились
  в top-10 E2. Результат согласуется с гипотезой, что символьные
  n-граммы устойчивее к словоформам и частичным совпадениям

- Для трех деклараций ранг ожидаемого регламента ухудшился,
  но он остался в top-3. Единственным пропуском E2 осталась
  декларация `D0085`

E2 становится основным лексическим вариантом

## E2b - BM25 lexical baseline

**Гипотеза:** BM25 может улучшить лексическое ранжирование благодаря
нормализации длины документа и ограничению вклада многократных
повторений одного слова

**Запрос:** `G31_1 + desc_extention`

**Документ:** `description`

**Модель:** Okapi BM25

**Токенизация:** слова в нижнем регистре, без лемматизации и удаления
стоп-слов

**Параметры:** `k1=1.5`, `b=0.75`

Параметры зафиксированы заранее и не подбираются на небольшой
dev-выборке. Остальные компоненты соответствуют предыдущим
лексическим экспериментам.

In [27]:
from rank_bm25 import BM25Okapi

In [28]:
BM25_TOKEN_PATTERN = re.compile(
    r"(?u)\b\w\w+\b"
)


def tokenize_bm25(text: str) -> list[str]:
    return BM25_TOKEN_PATTERN.findall(
        text.lower()
    )

In [29]:
def rank_regulations_bm25(
    declarations: pd.DataFrame,
    regulations: pd.DataFrame,
    top_k: int = 10,
    k1: float = 1.5,
    b: float = 0.75,
) -> pd.DataFrame:
    queries = (
        declarations["G31_1"].fillna("")
        + " "
        + declarations["desc_extention"].fillna("")
    )

    documents = regulations[
        "description"
    ].fillna("")

    tokenized_documents = [
        tokenize_bm25(text)
        for text in documents
    ]

    bm25 = BM25Okapi(
        tokenized_documents,
        k1=k1,
        b=b,
    )

    scores = np.vstack(
        [
            bm25.get_scores(
                tokenize_bm25(query)
            )
            for query in queries
        ]
    )

    top_indices = np.argsort(
        -scores,
        axis=1,
        kind="stable",
    )[:, :top_k]

    declaration_ids = declarations[
        "declaration_id"
    ].to_numpy()

    regulation_ids = regulations[
        "regulation_id"
    ].to_numpy()

    rows = []

    for query_index, regulation_indices in enumerate(
        top_indices
    ):
        for rank, regulation_index in enumerate(
            regulation_indices,
            start=1,
        ):
            rows.append(
                {
                    "declaration_id": declaration_ids[
                        query_index
                    ],
                    "rank": rank,
                    "regulation_id": regulation_ids[
                        regulation_index
                    ],
                    "score": float(
                        scores[
                            query_index,
                            regulation_index,
                        ]
                    ),
                }
            )

    return pd.DataFrame(
        rows,
        columns=PREDICTION_COLUMNS,
    )

In [30]:
started_at = perf_counter()

e2b_predictions = rank_regulations_bm25(
    declarations,
    regulations,
    top_k=10,
    k1=1.5,
    b=0.75,
)

e2b_elapsed = perf_counter() - started_at

validate_predictions(
    e2b_predictions,
    declarations,
    regulations,
    top_k=10,
)

e2b_metrics = calculate_ranking_metrics(
    e2b_predictions,
    dev_labels,
)

queries_without_matches = (
    e2b_predictions
    .groupby("declaration_id")["score"]
    .max()
    .eq(0)
    .sum()
)

print(f"Время E2b: {e2b_elapsed:.3f} секунды")
print(
    "Запросов без лексических совпадений: "
    f"{queries_without_matches}"
)

pd.Series(e2b_metrics)

Время E2b: 0.215 секунды
Запросов без лексических совпадений: 0


n_queries    15.000000
hit_at_1      0.333333
hit_at_5      0.733333
hit_at_10     0.800000
mrr_at_10     0.495079
dtype: float64

In [31]:
e2b_ranks = get_expected_ranks(
    e2b_predictions,
    dev_labels,
)

e2_e2b_comparison = dev_labels[
    [
        "declaration_id",
        "expected_regulation_id",
        "selection_type",
    ]
].copy()

e2_e2b_comparison["e2_rank"] = (
    e2_e2b_comparison["declaration_id"]
    .map(e2_ranks)
)

e2_e2b_comparison["e2b_rank"] = (
    e2_e2b_comparison["declaration_id"]
    .map(e2b_ranks)
)

e2_e2b_comparison["rank_change"] = (
    e2_e2b_comparison["e2_rank"]
    - e2_e2b_comparison["e2b_rank"]
)

display(
    e2_e2b_comparison.sort_values(
        "rank_change",
        ascending=False,
        na_position="last",
    )
)

,declaration_id,expected_regulation_id,selection_type,e2_rank,e2b_rank,rank_change
13,D0023,R0311,condition,3.0,1.0,2.0
6,D0012,R0133,random,2.0,1.0,1.0
10,D0082,R0182,numbers,2.0,1.0,1.0
0,D0032,R0019,random,1.0,1.0,0.0
12,D0088,R0219,materials,3.0,3.0,0.0
7,D0079,R0193,random,1.0,1.0,0.0
2,D0103,R0063,random,2.0,2.0,0.0
1,D0056,R0172,random,1.0,2.0,-1.0
3,D0119,R0228,random,3.0,4.0,-1.0
4,D0107,R0163,random,1.0,2.0,-1.0


### Результат E2b

- BM25 получил `Hit@1 = 0,333`, `Hit@5 = 0,733`,
  `Hit@10 = 0,800` и `MRR@10 = 0,495`. По всем основным
  метрикам он уступил E2

- Относительно E2 ранг ожидаемого регламента улучшился для
  трех запросов, не изменился для четырех и ухудшился для пяти.
  В трех случаях ожидаемый регламент не попал в top-10

- Для всех деклараций был найден хотя бы один документ
  с ненулевой оценкой BM25

Параметры BM25 дополнительно не подбираются из-за малого размера
dev-выборки. E2 остается основным лексическим вариантом

## E3 - комбинация word и char TF-IDF

**Гипотеза:** word TF-IDF учитывает точные словесные совпадения,
а char TF-IDF - совпадения частей слов. Их комбинация может улучшить
первые позиции, не снижая `Hit@10` E2

**Word-модель:** `analyzer="word"`, `ngram_range=(1, 1)`


**Char-модель:** `analyzer="char_wb"`, `ngram_range=(3, 5)`

**Комбинация:** среднее нормализованных оценок cosine similarity
с равными весами

**Остальные компоненты:** без изменений

In [32]:
started_at = perf_counter()

all_regulations_k = len(regulations)

word_all_predictions = rank_regulations(
    declarations,
    regulations,
    top_k=all_regulations_k,
    analyzer="word",
    ngram_range=(1, 1),
)

char_all_predictions = rank_regulations(
    declarations,
    regulations,
    top_k=all_regulations_k,
    analyzer="char_wb",
    ngram_range=(3, 5),
)

hybrid_scores = (
    word_all_predictions[
        [
            "declaration_id",
            "regulation_id",
            "score",
        ]
    ]
    .rename(columns={"score": "word_score"})
    .merge(
        char_all_predictions[
            [
                "declaration_id",
                "regulation_id",
                "score",
            ]
        ].rename(columns={"score": "char_score"}),
        on=[
            "declaration_id",
            "regulation_id",
        ],
        how="inner",
    )
)

In [33]:
hybrid_scores.head()

,declaration_id,regulation_id,word_score,char_score
0,D0001,R0003,0.469308,0.393323
1,D0001,R0002,0.289236,0.221322
2,D0001,R0001,0.271475,0.204247
3,D0001,R0074,0.158787,0.050374
4,D0001,R0026,0.148989,0.053024


In [34]:
word_max_scores = hybrid_scores.groupby(
    "declaration_id"
)["word_score"].transform("max")

char_max_scores = hybrid_scores.groupby(
    "declaration_id"
)["char_score"].transform("max")

hybrid_scores["word_score_normalized"] = (
    hybrid_scores["word_score"]
    / word_max_scores.where(word_max_scores.gt(0), 1.0)
)

hybrid_scores["char_score_normalized"] = (
    hybrid_scores["char_score"]
    / char_max_scores.where(char_max_scores.gt(0), 1.0)
)

In [35]:
WORD_WEIGHT = 0.5
CHAR_WEIGHT = 0.5

hybrid_scores["score"] = (
    WORD_WEIGHT
    * hybrid_scores["word_score_normalized"]
    + CHAR_WEIGHT
    * hybrid_scores["char_score_normalized"]
)

In [36]:
hybrid_scores["rank"] = (
    hybrid_scores
    .groupby("declaration_id")["score"]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)

e3_predictions = (
    hybrid_scores[
        hybrid_scores["rank"].le(10)
    ]
    [
        [
            "declaration_id",
            "rank",
            "regulation_id",
            "score",
        ]
    ]
    .sort_values(
        [
            "declaration_id",
            "rank",
        ]
    )
    .reset_index(drop=True)
)

e3_elapsed = perf_counter() - started_at

In [37]:
validate_predictions(
    e3_predictions,
    declarations,
    regulations,
    top_k=10,
)

e3_metrics = calculate_ranking_metrics(
    e3_predictions,
    dev_labels,
)

print(f"Время E3: {e3_elapsed:.3f} секунды")

pd.Series(e3_metrics)

Время E3: 7.612 секунды


n_queries    15.000000
hit_at_1      0.466667
hit_at_5      0.866667
hit_at_10     0.933333
mrr_at_10     0.651852
dtype: float64

In [38]:
experiment_results = pd.DataFrame(
    [
        {
            "experiment": "E0",
            "configuration": "word TF-IDF, (1, 1)",
            **e0_metrics,
        },
        {
            "experiment": "E1",
            "configuration": "word TF-IDF, (1, 2)",
            **e1_metrics,
        },
        {
            "experiment": "E2",
            "configuration": "char_wb TF-IDF, (3, 5)",
            **e2_metrics,
        },
        {
            "experiment": "E2b",
            "configuration": (
                "BM25Okapi, description, k1=1.5, b=0.75"
            ),
            **e2b_metrics,
        },
        {
            "experiment": "E3",
            "configuration": "word + char TF-IDF, 0.5 / 0.5",
            **e3_metrics,
        },
    ]
)

experiment_results

,experiment,configuration,n_queries,hit_at_1,hit_at_5,hit_at_10,mrr_at_10
0,E0,"word TF-IDF, (1, 1)",15,0.333333,0.800000,0.800000,0.487778
1,E1,"word TF-IDF, (1, 2)",15,0.333333,0.800000,0.800000,0.490000
2,E2,"char_wb TF-IDF, (3, 5)",15,0.466667,0.933333,0.933333,0.666667
3,E2b,"BM25Okapi, description, k1=1.5, b=0.75",15,0.333333,0.733333,0.800000,0.495079
4,E3,"word + char TF-IDF, 0.5 / 0.5",15,0.466667,0.866667,0.933333,0.651852


### Результат E3

- Комбинация word и char TF-IDF не улучшила E2. Значения `Hit@1`
и `Hit@10` не изменились, `Hit@5` снизился с 0,933 до 0,867,
а `MRR@10` - с 0,667 до 0,652

- Расчет занял около 7 секунд. E2 остается лучшим лексическим вариантом

## E4 - расширение текста регламента

**Гипотеза:** поля `notes` и `explanation` содержат условия применения,
исключения и ссылки на товарные категории, отсутствующие в кратком
`description`. Дополнительный контекст может помочь найти регламенты,
которые не попали в top-10 E2.

**Запрос:** `G31_1 + desc_extention`

**Варианты документов:**

- E4a - `description + explanation`
- E4b - `description + explanation + notes`

**Модель:** char TF-IDF

**Анализатор:** `char_wb`

**Диапазон n-грамм:** `(3, 5)`

**Остальные компоненты:** без изменений

### E4a - description + explanation

In [39]:
started_at = perf_counter()

e4a_predictions = rank_regulations(
    declarations,
    regulations,
    top_k=10,
    analyzer="char_wb",
    ngram_range=(3, 5),
    document_columns=(
        "description",
        "explanation",
    ),
)
e4a_elapsed = perf_counter() - started_at

validate_predictions(
    e4a_predictions,
    declarations,
    regulations,
    top_k=10,
)

e4a_metrics = calculate_ranking_metrics(
    e4a_predictions,
    dev_labels,
)

print(f"Время E4a: {e4a_elapsed:.3f} секунды")

pd.Series(e4a_metrics)

Время E4a: 0.665 секунды


n_queries    15.000000
hit_at_1      0.533333
hit_at_5      0.800000
hit_at_10     0.933333
mrr_at_10     0.642963
dtype: float64

### E4b - description + explanation + notes

In [40]:
started_at = perf_counter()

e4b_predictions = rank_regulations(
    declarations,
    regulations,
    top_k=10,
    analyzer="char_wb",
    ngram_range=(3, 5),
    document_columns=(
        "description",
        "explanation",
        "notes",
    ),
)

e4b_elapsed = perf_counter() - started_at

validate_predictions(
    e4b_predictions,
    declarations,
    regulations,
    top_k=10,
)

e4b_metrics = calculate_ranking_metrics(
    e4b_predictions,
    dev_labels,
)

print(f"Время E4b: {e4b_elapsed:.3f} секунды")

pd.Series(e4b_metrics)

Время E4b: 0.982 секунды


n_queries    15.000000
hit_at_1      0.466667
hit_at_5      0.800000
hit_at_10     1.000000
mrr_at_10     0.617222
dtype: float64

In [41]:
experiment_results = pd.DataFrame(
    [
        {
            "experiment": "E0",
            "configuration": "word TF-IDF, (1, 1)",
            **e0_metrics,
        },
        {
            "experiment": "E1",
            "configuration": "word TF-IDF, (1, 2)",
            **e1_metrics,
        },
        {
            "experiment": "E2",
            "configuration": "char_wb TF-IDF, (3, 5)",
            **e2_metrics,
        },
        {
            "experiment": "E2b",
            "configuration": (
                "BM25Okapi, description, k1=1.5, b=0.75"
            ),
            **e2b_metrics,
        },
        {
            "experiment": "E3",
            "configuration": "word + char TF-IDF, 0.5 / 0.5",
            **e3_metrics,
        },
       {
            "experiment": "E4a",
            "configuration": (
                "char_wb TF-IDF, description + explanation"
            ),
            **e4a_metrics,
        },
        {
            "experiment": "E4b",
            "configuration": (
                "char_wb TF-IDF, description + explanation + notes"
            ),
            **e4b_metrics,
        },
    ]
)

experiment_results

,experiment,configuration,n_queries,hit_at_1,hit_at_5,hit_at_10,mrr_at_10
0,E0,"word TF-IDF, (1, 1)",15,0.333333,0.800000,0.800000,0.487778
1,E1,"word TF-IDF, (1, 2)",15,0.333333,0.800000,0.800000,0.490000
2,E2,"char_wb TF-IDF, (3, 5)",15,0.466667,0.933333,0.933333,0.666667
3,E2b,"BM25Okapi, description, k1=1.5, b=0.75",15,0.333333,0.733333,0.800000,0.495079
4,E3,"word + char TF-IDF, 0.5 / 0.5",15,0.466667,0.866667,0.933333,0.651852
5,E4a,"char_wb TF-IDF, description + explanation",15,0.533333,0.800000,0.933333,0.642963
6,E4b,"char_wb TF-IDF, description + explanation + notes",15,0.466667,0.800000,1.000000,0.617222


## E4c - короткий путь по иерархии ТН ВЭД

**Гипотеза:** названия родительских категорий ТН ВЭД добавят краткий
контекст родительских категорий и помогут различать регламенты с короткими или
похожими описаниями

- Запрос: `G31_1 + desc_extention`
- Документ: `description + hierarchy_text`
- Путь: группа, товарная позиция и субпозиция — первые 2, 4 и 6 цифр кода
- Модель: `char_wb TF-IDF`, диапазон n-грамм `(3, 5)`
- Ранжирование выполняется по всем регламентам без жесткой фильтрации

Описание самого 10-значного кода не добавляется: его описание часто повторяет
основное описание регламента и не позволяет отдельно оценить пользу
родительского контекста

In [42]:
HIERARCHY_LINE_PATTERN = re.compile(
    r"^\s*(\d{2,10})\s*\|\s*(.+?)\s*$"
)


def load_hierarchy_nodes(path: str | Path) -> dict[str, str]:
    """Load code-to-description mapping from the TN VED hierarchy dump."""

    nodes = {}

    for line in Path(path).read_text(encoding="utf-8").splitlines():
        match = HIERARCHY_LINE_PATTERN.match(line)

        if not match:
            continue

        code, description = match.groups()

        description = re.sub(
            r"^[-–—]+\s*",
            "",
            description,
        ).strip().rstrip(":")

        nodes.setdefault(code, description)

    return nodes


def build_short_hierarchy_path(
    code: str,
    nodes: dict[str, str],
) -> str:
    """Build a path from 2-, 4- and 6-digit parent descriptions."""

    parts = []

    for prefix_length in (2, 4, 6):
        description = nodes.get(code[:prefix_length])

        if description and description not in parts:
            parts.append(description)

    return " ".join(parts)

In [43]:
hierarchy_nodes = load_hierarchy_nodes(
    "data/tnved_knowledge.txt"
)

regulations_e4c = regulations.copy()

regulations_e4c["hierarchy_text"] = (
    regulations_e4c["code"]
    .astype(str)
    .map(
        lambda code: build_short_hierarchy_path(
            code,
            hierarchy_nodes,
        )
    )
)

hierarchy_coverage = pd.Series(
    {
        "group_2": int(
            regulations_e4c["code"]
            .str[:2]
            .isin(hierarchy_nodes)
            .sum()
        ),
        "heading_4": int(
            regulations_e4c["code"]
            .str[:4]
            .isin(hierarchy_nodes)
            .sum()
        ),
        "subheading_6": int(
            regulations_e4c["code"]
            .str[:6]
            .isin(hierarchy_nodes)
            .sum()
        ),
        "non_empty_path": int(
            regulations_e4c["hierarchy_text"]
            .ne("")
            .sum()
        ),
    },
    name="regulation_count",
)

display(hierarchy_coverage)

display(
    regulations_e4c[
        [
            "code",
            "description",
            "hierarchy_text",
        ]
    ].head()
)

assert regulations_e4c["hierarchy_text"].ne("").all()

group_2           360
heading_4         351
subheading_6      230
non_empty_path    360
Name: regulation_count, dtype: int64

,code,description,hierarchy_text
0,3004500006,Лекарственные средства (кроме товаров товарной...,ФАРМАЦЕВТИЧЕСКАЯ ПРОДУКЦИЯ Лекарственные средс...
1,8407343009,Двигатели внутреннего сгорания с искровым зажи...,"РЕАКТОРЫ ЯДЕРНЫЕ, КОТЛЫ, ОБОРУДОВАНИЕ И МЕХАНИ..."
2,2933191000,Фенилбутазон (inn),ОРГАНИЧЕСКИЕ ХИМИЧЕСКИЕ СОЕДИНЕНИЯ Соединения ...
3,9110119000,"Прочие укомплектованные механизмы часовые, нес...",ЧАСЫ ВСЕХ ВИДОВ И ИХ ЧАСТИ Укомплектованные ме...
4,6203433100,Комбинезоны с нагрудниками и лямками производс...,"ПРЕДМЕТЫ ОДЕЖДЫ И ПРИНАДЛЕЖНОСТИ К ОДЕЖДЕ, КРО..."


In [44]:
e4c_started_at = perf_counter()

e4c_predictions = rank_regulations(
    declarations,
    regulations_e4c,
    top_k=10,
    analyzer="char_wb",
    ngram_range=(3, 5),
    document_columns=(
        "description",
        "hierarchy_text",
    ),
)

e4c_elapsed = perf_counter() - e4c_started_at

validate_predictions(
    e4c_predictions,
    declarations,
    regulations_e4c,
    top_k=10,
)

e4c_metrics = calculate_ranking_metrics(
    e4c_predictions,
    dev_labels,
)

print(f"Время E4c: {e4c_elapsed:.3f} секунды")

pd.Series(e4c_metrics)

Время E4c: 0.454 секунды


n_queries    15.000000
hit_at_1      0.533333
hit_at_5      0.866667
hit_at_10     0.933333
mrr_at_10     0.652222
dtype: float64

In [45]:
e4c_ranks = get_expected_ranks(
    e4c_predictions,
    dev_labels,
)

e2_e4c_comparison = dev_labels[
    [
        "declaration_id",
        "expected_regulation_id",
        "selection_type",
    ]
].copy()

e2_e4c_comparison["e2_rank"] = (
    e2_e4c_comparison["declaration_id"]
    .map(e2_ranks)
)

e2_e4c_comparison["e4c_rank"] = (
    e2_e4c_comparison["declaration_id"]
    .map(e4c_ranks)
)

e2_e4c_comparison["change_vs_e2"] = (
    e2_e4c_comparison["e2_rank"]
    - e2_e4c_comparison["e4c_rank"]
)

display(
    e2_e4c_comparison.sort_values(
        "change_vs_e2",
        ascending=False,
        na_position="last",
    )
)

,declaration_id,expected_regulation_id,selection_type,e2_rank,e4c_rank,change_vs_e2
13,D0023,R0311,condition,3.0,1.0,2.0
10,D0082,R0182,numbers,2.0,1.0,1.0
4,D0107,R0163,random,1.0,1.0,0.0
6,D0012,R0133,random,2.0,2.0,0.0
1,D0056,R0172,random,1.0,1.0,0.0
12,D0088,R0219,materials,3.0,3.0,0.0
11,D0099,R0052,negation,1.0,1.0,0.0
8,D0043,R0308,random,1.0,1.0,0.0
7,D0079,R0193,random,1.0,1.0,0.0
14,D0040,R0127,low_lexical_score,1.0,1.0,0.0


### Вывод по E4

- Добавление `explanation` повысило `Hit@1` с 0,467 до 0,533, но
снизило `Hit@5` с 0,933 до 0,800 и `MRR@10` с 0,667 до 0,643

- Добавление `notes` повысило `Hit@10` до 1,000 и восстановило
пропущенный E2 регламент. При этом `Hit@5` снизился до 0,800,
а `MRR@10` - до 0,617

- Добавление короткого пути ТН ВЭД повысило `Hit@1` до 0,533,
но снизило `Hit@5` до 0,867 и `MRR@10` до 0,652.
Один пропущенный регламент был восстановлен, но другой выпал из top-10

Дополнительный контекст улучшил отдельные запросы, но одновременно
снизил другие метрики. Поэтому E2 с одним полем `description`
остается основным лексическим ранжировщиком

## Проверка E4 на high-confidence части

In [46]:
high_confidence_dev = dev_labels.query(
    "confidence == 'high'"
)

pd.DataFrame(
    [
        {
            "experiment": "E2",
            **calculate_ranking_metrics(
                e2_predictions,
                high_confidence_dev,
            ),
        },
        {
            "experiment": "E4a",
            **calculate_ranking_metrics(
                e4a_predictions,
                high_confidence_dev,
            ),
        },
        {
            "experiment": "E4b",
            **calculate_ranking_metrics(
                e4b_predictions,
                high_confidence_dev,
            ),
        },
        {
            "experiment": "E4c",
            **calculate_ranking_metrics(
                e4c_predictions,
                high_confidence_dev
            ),
        },
    ]
)

,experiment,n_queries,hit_at_1,hit_at_5,hit_at_10,mrr_at_10
0,E2,13,0.538462,1.000000,1.000000,0.743590
1,E4a,13,0.615385,0.846154,0.923077,0.707692
2,E4b,13,0.538462,0.846154,1.000000,0.678846
3,E4c,13,0.615385,0.846154,0.923077,0.707692


### Вывод по уверенным меткам

- На 13 примерах с высокой уверенностью E2 получил `Hit@5` и `Hit@10`, равные 1,000, и максимальный `MRR@10 = 0,744`

- E4a и E4c повысили `Hit@1` до 0,615, но снизили `Hit@10` до 0,923 и `MRR@10` до 0,708

- Преимущество E4b по `Hit@10` на полной dev-выборке связано
  с `D0085` - примером со средней уверенностью

Проверка подтверждает выбор E2 как наиболее устойчивой лексической конфигурации

## Промежуточная таблица лексических экспериментов

In [47]:
experiment_results = pd.DataFrame(
    [
        {
            "experiment": "E0",
            "configuration": "word TF-IDF, (1, 1)",
            **e0_metrics,
        },
        {
            "experiment": "E1",
            "configuration": "word TF-IDF, (1, 2)",
            **e1_metrics,
        },
        {
            "experiment": "E2",
            "configuration": "char_wb TF-IDF, (3, 5)",
            **e2_metrics,
        },
        {
            "experiment": "E2b",
            "configuration": (
                "BM25Okapi, description, k1=1.5, b=0.75"
            ),
            **e2b_metrics,
        },
        {
            "experiment": "E3",
            "configuration": "word + char TF-IDF, 0.5 / 0.5",
            **e3_metrics,
        },
       {
            "experiment": "E4a",
            "configuration": (
                "char_wb TF-IDF, description + explanation"
            ),
            **e4a_metrics,
        },
        {
            "experiment": "E4b",
            "configuration": (
                "char_wb TF-IDF, description + explanation + notes"
            ),
            **e4b_metrics,
        },
        {
            "experiment": "E4c",
            "configuration": (
                "char_wb TF-IDF, description + short hierarchy path"
            ),
            **e4c_metrics,
        },
    ]
)

display(experiment_results.round(3))

,experiment,configuration,n_queries,hit_at_1,hit_at_5,hit_at_10,mrr_at_10
0,E0,"word TF-IDF, (1, 1)",15,0.333,0.800,0.800,0.488
1,E1,"word TF-IDF, (1, 2)",15,0.333,0.800,0.800,0.490
2,E2,"char_wb TF-IDF, (3, 5)",15,0.467,0.933,0.933,0.667
3,E2b,"BM25Okapi, description, k1=1.5, b=0.75",15,0.333,0.733,0.800,0.495
4,E3,"word + char TF-IDF, 0.5 / 0.5",15,0.467,0.867,0.933,0.652
5,E4a,"char_wb TF-IDF, description + explanation",15,0.533,0.800,0.933,0.643
6,E4b,"char_wb TF-IDF, description + explanation + notes",15,0.467,0.800,1.000,0.617
7,E4c,"char_wb TF-IDF, description + short hierarchy ...",15,0.533,0.867,0.933,0.652


### Промежуточный выбор лексической модели

- Лучшей самостоятельной лексической конфигурацией выбран E2:
`char_wb TF-IDF` с диапазоном n-грамм `(3, 5)` и документом
из поля `description`

- E2 получил максимальные `MRR@10` и `Hit@5` среди лексических
вариантов и нашел все ожидаемые регламенты в top-5 на
high-confidence части выборки

- Расширение текста полями `explanation`, `notes` и коротким путем
ТН ВЭД не дало устойчивого улучшения. Следующий эксперимент
проверяет, может ли семантическая модель дополнить E2

## E5 - мультиязычные эмбеддинги E5

В этом эксперименте TF-IDF заменяется моделью
`intfloat/multilingual-e5-small`

Для контролируемого сравнения с E2 состав текстов не меняется:

- запрос - `G31_1 + desc_extention`
- документ - `description`
- ранжирование - cosine similarity
- результат - top-10

К запросам добавляется префикс `query:`, к документам - `passage:`.
Эмбеддинги нормализуются, поэтому для расчета сходства используется
скалярное произведение

Модель и ее ревизия фиксируются до расчета dev-метрик.
Holdout в эксперименте не используется

In [48]:
from sentence_transformers import SentenceTransformer

In [49]:
MODEL_NAME = "intfloat/multilingual-e5-small"
MODEL_REVISION = "d1d99a1efae6779390caba937d92c54b5bc70e51"

load_started_at = perf_counter()

e5_model = SentenceTransformer(
    MODEL_NAME,
    revision=MODEL_REVISION,
    device="cpu",
)

model_load_time = perf_counter() - load_started_at

print(f"Загрузка модели: {model_load_time:.3f} секунды")

Загрузка модели: 2.959 секунды


In [50]:
e5_query_texts = (
    "query: "
    + declarations["G31_1"].fillna("")
    + " "
    + declarations["desc_extention"].fillna("")
).tolist()

e5_document_texts = (
    "passage: "
    + regulations["description"].fillna("")
).tolist()

print(f"Запросов: {len(e5_query_texts)}")
print(f"Документов: {len(e5_document_texts)}")

Запросов: 120
Документов: 360


In [51]:
e5_started_at = perf_counter()

query_embeddings = e5_model.encode(
    e5_query_texts,
    batch_size=32,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False,
)

document_embeddings = e5_model.encode(
    e5_document_texts,
    batch_size=32,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False,
)

e5_similarities = query_embeddings @ document_embeddings.T

top_k = 10

top_indices = np.argsort(
    -e5_similarities,
    axis=1,
    kind="stable",
)[:, :top_k]

declaration_ids = declarations["declaration_id"].to_numpy()
regulation_ids = regulations["regulation_id"].to_numpy()

rows = []

for query_index, candidate_indices in enumerate(top_indices):
    for rank, regulation_index in enumerate(candidate_indices, start=1):
        rows.append(
            {
                "declaration_id": declaration_ids[query_index],
                "rank": rank,
                "regulation_id": regulation_ids[regulation_index],
                "score": float(
                    e5_similarities[query_index, regulation_index]
                ),
            }
        )

e5_predictions = pd.DataFrame(rows)

e5_time = perf_counter() - e5_started_at

print(f"Время E5 без загрузки модели: {e5_time:.3f} секунды")
print(f"Размер эмбеддингов запросов: {query_embeddings.shape}")
print(f"Размер эмбеддингов документов: {document_embeddings.shape}")

display(e5_predictions.head(10))

Время E5 без загрузки модели: 14.687 секунды
Размер эмбеддингов запросов: (120, 384)
Размер эмбеддингов документов: (360, 384)


,declaration_id,rank,regulation_id,score
0,D0001,1,R0003,0.853199
1,D0001,2,R0002,0.849416
2,D0001,3,R0001,0.847449
3,D0001,4,R0119,0.841845
4,D0001,5,R0120,0.838845
5,D0001,6,R0035,0.838218
6,D0001,7,R0306,0.836405
7,D0001,8,R0033,0.836400
8,D0001,9,R0128,0.835420
9,D0001,10,R0167,0.835019


In [52]:
validate_predictions(
    e5_predictions,
    declarations,
    regulations,
    top_k=10,
)

e5_metrics = calculate_ranking_metrics(
    e5_predictions,
    dev_labels,
    ks=(1, 5, 10),
)

pd.Series(e5_metrics)

n_queries    15.000000
hit_at_1      0.400000
hit_at_5      1.000000
hit_at_10     1.000000
mrr_at_10     0.641111
dtype: float64

In [53]:
e5_result = {
    "experiment": "E5",
    "configuration": "multilingual-e5-small, description",
    **e5_metrics,
}

experiment_results = pd.concat(
    [
        experiment_results,
        pd.DataFrame([e5_result]),
    ],
    ignore_index=True,
)

experiment_results

,experiment,configuration,n_queries,hit_at_1,hit_at_5,hit_at_10,mrr_at_10
0,E0,"word TF-IDF, (1, 1)",15,0.333333,0.800000,0.800000,0.487778
1,E1,"word TF-IDF, (1, 2)",15,0.333333,0.800000,0.800000,0.490000
2,E2,"char_wb TF-IDF, (3, 5)",15,0.466667,0.933333,0.933333,0.666667
3,E2b,"BM25Okapi, description, k1=1.5, b=0.75",15,0.333333,0.733333,0.800000,0.495079
4,E3,"word + char TF-IDF, 0.5 / 0.5",15,0.466667,0.866667,0.933333,0.651852
5,E4a,"char_wb TF-IDF, description + explanation",15,0.533333,0.800000,0.933333,0.642963
6,E4b,"char_wb TF-IDF, description + explanation + notes",15,0.466667,0.800000,1.000000,0.617222
7,E4c,"char_wb TF-IDF, description + short hierarchy ...",15,0.533333,0.866667,0.933333,0.652222
8,E5,"multilingual-e5-small, description",15,0.400000,1.000000,1.000000,0.641111


In [54]:
e5_ranks = get_expected_ranks(
    e5_predictions,
    dev_labels,
)

e2_e5_comparison = (
    dev_labels[
        [
            "declaration_id",
            "expected_regulation_id",
            "selection_type",
        ]
    ]
    .merge(
        e2_ranks.rename("e2_rank"),
        on="declaration_id",
        how="left",
    )
    .merge(
        e5_ranks.rename("e5_rank"),
        on="declaration_id",
        how="left",
    )
)

e2_e5_comparison["rank_change"] = (
    e2_e5_comparison["e2_rank"]
    - e2_e5_comparison["e5_rank"]
)

display(
    e2_e5_comparison.sort_values(
        "rank_change",
        ascending=False,
        na_position="last",
    )
)

,declaration_id,expected_regulation_id,selection_type,e2_rank,e5_rank,rank_change
13,D0023,R0311,condition,3.0,1,2.0
10,D0082,R0182,numbers,2.0,1,1.0
6,D0012,R0133,random,2.0,2,0.0
3,D0119,R0228,random,3.0,3,0.0
1,D0056,R0172,random,1.0,1,0.0
2,D0103,R0063,random,2.0,2,0.0
9,D0064,R0179,random,2.0,2,0.0
12,D0088,R0219,materials,3.0,3,0.0
8,D0043,R0308,random,1.0,1,0.0
7,D0079,R0193,random,1.0,1,0.0


### Результат E5

- По сравнению с E2 значение `Hit@1` снизилось с 0,467 до 0,400,
  а `MRR@10` с 0,667 до 0,641. При этом E5 получила
  `Hit@5 = Hit@10 = 1,000`

- E5 вернула `R0268` для декларации `D0085`, отсутствовавший
  в top-10 E2. Для `D0099` ожидаемый регламент, наоборот,
  опустился с первого на четвертое место

E5 не превосходит E2 как самостоятельный ранжировщик, но может
служить вторым источником кандидатов. Это проверяется в E6

## E6 - объединение char TF-IDF и multilingual E5

**Гипотеза:** E2 и E5 допускают разные ошибки. Объединение их рангов
может сохранить верхние позиции E2 и добавить кандидатов,
найденных только E5

**Компоненты:**

- E2 - `char_wb TF-IDF`, `(3, 5)`
- E5 - `multilingual-e5-small`
- документ - `description`
- объединение - Reciprocal Rank Fusion
- параметр RRF - `k=60`
- глубина объединения - все регламенты
- результат top-10

RRF объединяет позиции кандидатов по формуле `1 / (k + rank)`
и не требует сравнивать оценки TF-IDF и E5 напрямую

Параметр `k=60` фиксируется заранее и не подбирается на dev.
При одинаковой оценке RRF выше ставится кандидат с лучшим рангом E2

In [55]:
def predictions_from_scores(
    scores: np.ndarray,
    declarations: pd.DataFrame,
    regulations: pd.DataFrame,
    top_k: int,
) -> pd.DataFrame:
    expected_shape = (
        len(declarations),
        len(regulations),
    )

    if scores.shape != expected_shape:
        raise ValueError(
            f"scores must have shape {expected_shape}"
        )

    top_indices = np.argsort(
        -scores,
        axis=1,
        kind="stable",
    )[:, :top_k]

    declaration_ids = declarations[
        "declaration_id"
    ].to_numpy()

    regulation_ids = regulations[
        "regulation_id"
    ].to_numpy()

    rows = []

    for query_index, regulation_indices in enumerate(
        top_indices
    ):
        for rank, regulation_index in enumerate(
            regulation_indices,
            start=1,
        ):
            rows.append(
                {
                    "declaration_id": declaration_ids[
                        query_index
                    ],
                    "rank": rank,
                    "regulation_id": regulation_ids[
                        regulation_index
                    ],
                    "score": float(
                        scores[
                            query_index,
                            regulation_index,
                        ]
                    ),
                }
            )

    return pd.DataFrame(
        rows,
        columns=PREDICTION_COLUMNS,
    )

In [56]:
candidate_count = len(regulations)

e2_full_started_at = perf_counter()

e2_full_predictions = rank_regulations(
    declarations,
    regulations,
    top_k=candidate_count,
    analyzer="char_wb",
    ngram_range=(3, 5),
)

e2_full_time = (
    perf_counter() - e2_full_started_at
)

e5_full_started_at = perf_counter()

e5_full_predictions = predictions_from_scores(
    e5_similarities,
    declarations,
    regulations,
    top_k=candidate_count,
)

e5_full_ranking_time = (
    perf_counter() - e5_full_started_at
)

validate_predictions(
    e2_full_predictions,
    declarations,
    regulations,
    top_k=candidate_count,
)

validate_predictions(
    e5_full_predictions,
    declarations,
    regulations,
    top_k=candidate_count,
)

print(
    "Полное ранжирование E2: "
    f"{e2_full_time:.3f} секунды"
)

print(
    "Построение полного ранжирования E5 "
    "из готовой матрицы: "
    f"{e5_full_ranking_time:.3f} секунды"
)

Полное ранжирование E2: 3.887 секунды
Построение полного ранжирования E5 из готовой матрицы: 0.045 секунды


In [57]:
def reciprocal_rank_fusion(
    e2_predictions: pd.DataFrame,
    e5_predictions: pd.DataFrame,
    top_k: int = 10,
    rrf_k: int = 60,
) -> pd.DataFrame:
    e2_ranking = (
        e2_predictions[
            [
                "declaration_id",
                "regulation_id",
                "rank",
            ]
        ]
        .rename(
            columns={
                "rank": "e2_rank",
            }
        )
    )

    e5_ranking = (
        e5_predictions[
            [
                "declaration_id",
                "regulation_id",
                "rank",
            ]
        ]
        .rename(
            columns={
                "rank": "e5_rank",
            }
        )
    )

    fused = e2_ranking.merge(
        e5_ranking,
        on=[
            "declaration_id",
            "regulation_id",
        ],
        how="inner",
        validate="one_to_one",
    )

    fused["score"] = (
        1 / (rrf_k + fused["e2_rank"])
        + 1 / (rrf_k + fused["e5_rank"])
    )

    fused = fused.sort_values(
        [
            "declaration_id",
            "score",
            "e2_rank",
            "e5_rank",
            "regulation_id",
        ],
        ascending=[
            True,
            False,
            True,
            True,
            True,
        ],
        kind="stable",
    )

    fused["rank"] = (
        fused
        .groupby(
            "declaration_id",
            sort=False,
        )
        .cumcount()
        + 1
    )

    return (
        fused.loc[
            fused["rank"].le(top_k),
            PREDICTION_COLUMNS,
        ]
        .reset_index(drop=True)
    )

In [58]:
RRF_K = 60

e6_started_at = perf_counter()

e6_predictions = reciprocal_rank_fusion(
    e2_full_predictions,
    e5_full_predictions,
    top_k=10,
    rrf_k=RRF_K,
)

e6_fusion_time = (
    perf_counter() - e6_started_at
)

validate_predictions(
    e6_predictions,
    declarations,
    regulations,
    top_k=10,
)

e6_metrics = calculate_ranking_metrics(
    e6_predictions,
    dev_labels,
    ks=(1, 5, 10),
)

print(
    f"Время RRF-объединения: "
    f"{e6_fusion_time:.3f} секунды"
)

pd.Series(e6_metrics)

Время RRF-объединения: 0.133 секунды


n_queries    15.000000
hit_at_1      0.533333
hit_at_5      0.933333
hit_at_10     1.000000
mrr_at_10     0.722222
dtype: float64

In [59]:
e6_result = {
    "experiment": "E6",
    "configuration": (
        "RRF: char_wb TF-IDF + "
        "multilingual-e5-small, k=60"
    ),
    **e6_metrics,
}

experiment_results = pd.concat(
    [
        experiment_results.loc[
            ~experiment_results[
                "experiment"
            ].eq("E6")
        ],
        pd.DataFrame([e6_result]),
    ],
    ignore_index=True,
)

display(
    experiment_results.round(3)
)

,experiment,configuration,n_queries,hit_at_1,hit_at_5,hit_at_10,mrr_at_10
0,E0,"word TF-IDF, (1, 1)",15,0.333,0.800,0.800,0.488
1,E1,"word TF-IDF, (1, 2)",15,0.333,0.800,0.800,0.490
2,E2,"char_wb TF-IDF, (3, 5)",15,0.467,0.933,0.933,0.667
3,E2b,"BM25Okapi, description, k1=1.5, b=0.75",15,0.333,0.733,0.800,0.495
4,E3,"word + char TF-IDF, 0.5 / 0.5",15,0.467,0.867,0.933,0.652
5,E4a,"char_wb TF-IDF, description + explanation",15,0.533,0.800,0.933,0.643
6,E4b,"char_wb TF-IDF, description + explanation + notes",15,0.467,0.800,1.000,0.617
7,E4c,"char_wb TF-IDF, description + short hierarchy ...",15,0.533,0.867,0.933,0.652
8,E5,"multilingual-e5-small, description",15,0.400,1.000,1.000,0.641
9,E6,"RRF: char_wb TF-IDF + multilingual-e5-small, k=60",15,0.533,0.933,1.000,0.722


In [60]:
e2_dev_ranks = get_expected_ranks(
    e2_predictions,
    dev_labels,
)

e5_dev_ranks = get_expected_ranks(
    e5_predictions,
    dev_labels,
)

e6_dev_ranks = get_expected_ranks(
    e6_predictions,
    dev_labels,
)

e6_rank_comparison = dev_labels[
    [
        "declaration_id",
        "expected_regulation_id",
        "selection_type",
    ]
].copy()

e6_rank_comparison["e2_rank"] = (
    e6_rank_comparison["declaration_id"]
    .map(e2_dev_ranks)
)

e6_rank_comparison["e5_rank"] = (
    e6_rank_comparison["declaration_id"]
    .map(e5_dev_ranks)
)

e6_rank_comparison["e6_rank"] = (
    e6_rank_comparison["declaration_id"]
    .map(e6_dev_ranks)
)

e6_rank_comparison["change_vs_e2"] = (
    e6_rank_comparison["e2_rank"]
    - e6_rank_comparison["e6_rank"]
)

display(
    e6_rank_comparison.sort_values(
        "change_vs_e2",
        ascending=False,
        na_position="last",
    )
)

,declaration_id,expected_regulation_id,selection_type,e2_rank,e5_rank,e6_rank,change_vs_e2
13,D0023,R0311,condition,3.0,1,2,1.0
10,D0082,R0182,numbers,2.0,1,1,1.0
2,D0103,R0063,random,2.0,2,2,0.0
3,D0119,R0228,random,3.0,3,3,0.0
0,D0032,R0019,random,1.0,2,1,0.0
1,D0056,R0172,random,1.0,1,1,0.0
6,D0012,R0133,random,2.0,2,2,0.0
4,D0107,R0163,random,1.0,2,1,0.0
8,D0043,R0308,random,1.0,1,1,0.0
7,D0079,R0193,random,1.0,1,1,0.0


### Результат E6

- E6 получил `Hit@1 = 0,533`, `Hit@5 = 0,933`,
  `Hit@10 = 1,000` и `MRR@10 = 0,722`. Это лучший
  результат на dev среди проверенных конфигураций

- Относительно E2 ожидаемый регламент для `D0023` поднялся
  с третьего на второе место, а для `D0082` - со второго
  на первое. Регламент `R0268` для `D0085` появился
  на шестом месте

- Позиции остальных ожидаемых регламентов относительно E2
  не ухудшились

При финальном прогоне полный расчет E2 занял около 3,9 секунды,
загрузка E5 - 3,0 секунды, построение эмбеддингов - 14,7 секунды,
а объединение RRF - 0,13 секунды

E6 становится основным кандидатом перед проверкой реранкера

## E7 - cross-encoder reranking

**Гипотеза:** cross-encoder совместно обрабатывает текст запроса
и документа, поэтому может улучшить порядок top-30 кандидатов E6

**Генератор кандидатов:** E6 - RRF из E2 и E5

**Глубина кандидатов:** top-30

**Реранкер:** `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`

**Запрос:** `G31_1 + desc_extention`

**Документ:** `description`

**Результат:** top-10 по оценке cross-encoder

Модель используется без дообучения. Глубина кандидатов, модель
и остальные параметры фиксируются до оценки на dev.
Holdout в эксперименте не используется

Максимальная длина входной последовательности - 512 токенов.
Выход модели используется только как оценка для сортировки
и не интерпретируется как вероятность

In [61]:
from sentence_transformers import CrossEncoder

In [62]:
E7_CANDIDATE_K = 30
E7_TOP_K = 10

e6_candidate_predictions = reciprocal_rank_fusion(
    e2_full_predictions,
    e5_full_predictions,
    top_k=E7_CANDIDATE_K,
    rrf_k=RRF_K,
)

validate_predictions(
    e6_candidate_predictions,
    declarations,
    regulations,
    top_k=E7_CANDIDATE_K,
)

e7_candidates = e6_candidate_predictions.rename(
    columns={
        "rank": "e6_rank",
        "score": "e6_score",
    }
)

assert len(e7_candidates) == (
    len(declarations) * E7_CANDIDATE_K
)

display(e7_candidates.head())

,declaration_id,e6_rank,regulation_id,e6_score
0,D0001,1,R0003,0.032787
1,D0001,2,R0002,0.032258
2,D0001,3,R0001,0.031746
3,D0001,4,R0226,0.023960
4,D0001,5,R0120,0.023515


In [63]:
RERANKER_NAME = (
    "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
)

RERANKER_REVISION = (
    "1427fd652930e4ba29e8149678df786c240d8825"
)

reranker_load_started_at = perf_counter()

reranker = CrossEncoder(
    RERANKER_NAME,
    revision=RERANKER_REVISION,
    max_length=512,
    device="cpu",
)

reranker_load_time = (
    perf_counter() - reranker_load_started_at
)

print(
    "Загрузка реранкера: "
    f"{reranker_load_time:.3f} секунды"
)

Загрузка реранкера: 1.496 секунды


In [64]:
e7_query_texts = (
    declarations
    .set_index("declaration_id")[
        [
            "G31_1",
            "desc_extention",
        ]
    ]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.strip()
)

e7_document_texts = (
    regulations
    .set_index("regulation_id")[
        "description"
    ]
    .fillna("")
)

e7_pairs = [
    (
        str(
            e7_query_texts.loc[
                row.declaration_id
            ]
        ),
        str(
            e7_document_texts.loc[
                row.regulation_id
            ]
        ),
    )
    for row in e7_candidates.itertuples(
        index=False
    )
]

assert len(e7_pairs) == len(e7_candidates)

print(f"Пар для реранжирования: {len(e7_pairs)}")

Пар для реранжирования: 3600


In [65]:
RERANKER_BATCH_SIZE = 16

e7_started_at = perf_counter()

e7_scores = reranker.predict(
    e7_pairs,
    batch_size=RERANKER_BATCH_SIZE,
    show_progress_bar=False,
    convert_to_numpy=True,
)

e7_elapsed = perf_counter() - e7_started_at

e7_scores = np.asarray(
    e7_scores,
    dtype=float,
).reshape(-1)

assert len(e7_scores) == len(e7_candidates)
assert np.isfinite(e7_scores).all()

print(
    "Время реранжирования: "
    f"{e7_elapsed:.3f} секунды"
)

Время реранжирования: 361.377 секунды


In [66]:
e7_scored_candidates = e7_candidates.copy()

e7_scored_candidates["score"] = e7_scores

e7_scored_candidates = (
    e7_scored_candidates
    .sort_values(
        [
            "declaration_id",
            "score",
            "e6_rank",
            "regulation_id",
        ],
        ascending=[
            True,
            False,
            True,
            True,
        ],
        kind="stable",
    )
)

e7_scored_candidates["rank"] = (
    e7_scored_candidates
    .groupby(
        "declaration_id",
        sort=False,
    )
    .cumcount()
    .add(1)
)

e7_predictions = (
    e7_scored_candidates.loc[
        e7_scored_candidates["rank"].le(
            E7_TOP_K
        ),
        PREDICTION_COLUMNS,
    ]
    .reset_index(drop=True)
)

validate_predictions(
    e7_predictions,
    declarations,
    regulations,
    top_k=E7_TOP_K,
)

display(e7_predictions.head(10))

,declaration_id,rank,regulation_id,score
0,D0001,1,R0001,-0.062355
1,D0001,2,R0002,-0.067069
2,D0001,3,R0003,-1.215607
3,D0001,4,R0064,-2.127848
4,D0001,5,R0019,-2.720560
5,D0001,6,R0021,-2.725405
6,D0001,7,R0226,-3.042799
7,D0001,8,R0026,-3.077507
8,D0001,9,R0027,-3.156973
9,D0001,10,R0254,-3.237730


In [67]:
e7_metrics = calculate_ranking_metrics(
    e7_predictions,
    dev_labels,
    ks=(1, 5, 10),
)

pd.Series(e7_metrics)

n_queries    15.000000
hit_at_1      0.400000
hit_at_5      0.933333
hit_at_10     0.933333
mrr_at_10     0.602222
dtype: float64

## Итоговая таблица экспериментов

In [68]:
e7_result = {
    "experiment": "E7",
    "configuration": (
        "E6 top-30 + multilingual "
        "MMARCO cross-encoder"
    ),
    **e7_metrics,
}

experiment_results = pd.concat(
    [
        experiment_results.loc[
            ~experiment_results[
                "experiment"
            ].eq("E7")
        ],
        pd.DataFrame([e7_result]),
    ],
    ignore_index=True,
)

display(experiment_results.round(3))

,experiment,configuration,n_queries,hit_at_1,hit_at_5,hit_at_10,mrr_at_10
0,E0,"word TF-IDF, (1, 1)",15,0.333,0.800,0.800,0.488
1,E1,"word TF-IDF, (1, 2)",15,0.333,0.800,0.800,0.490
2,E2,"char_wb TF-IDF, (3, 5)",15,0.467,0.933,0.933,0.667
3,E2b,"BM25Okapi, description, k1=1.5, b=0.75",15,0.333,0.733,0.800,0.495
4,E3,"word + char TF-IDF, 0.5 / 0.5",15,0.467,0.867,0.933,0.652
5,E4a,"char_wb TF-IDF, description + explanation",15,0.533,0.800,0.933,0.643
6,E4b,"char_wb TF-IDF, description + explanation + notes",15,0.467,0.800,1.000,0.617
7,E4c,"char_wb TF-IDF, description + short hierarchy ...",15,0.533,0.867,0.933,0.652
8,E5,"multilingual-e5-small, description",15,0.400,1.000,1.000,0.641
9,E6,"RRF: char_wb TF-IDF + multilingual-e5-small, k=60",15,0.533,0.933,1.000,0.722


In [69]:
e7_ranks = get_expected_ranks(
    e7_predictions,
    dev_labels,
)

e6_e7_comparison = dev_labels[
    [
        "declaration_id",
        "expected_regulation_id",
        "selection_type",
        "confidence",
    ]
].copy()

e6_e7_comparison["e6_rank"] = (
    e6_e7_comparison["declaration_id"]
    .map(e6_dev_ranks)
)

e6_e7_comparison["e7_rank"] = (
    e6_e7_comparison["declaration_id"]
    .map(e7_ranks)
)

e6_e7_comparison["change_vs_e6"] = (
    e6_e7_comparison["e6_rank"]
    - e6_e7_comparison["e7_rank"]
)

display(
    e6_e7_comparison.sort_values(
        "change_vs_e6",
        ascending=False,
        na_position="last",
    )
)

,declaration_id,expected_regulation_id,selection_type,confidence,e6_rank,e7_rank,change_vs_e6
2,D0103,R0063,random,high,2,1.0,1.0
0,D0032,R0019,random,high,1,1.0,0.0
4,D0107,R0163,random,high,1,1.0,0.0
3,D0119,R0228,random,medium,3,3.0,0.0
9,D0064,R0179,random,high,2,2.0,0.0
8,D0043,R0308,random,high,1,1.0,0.0
14,D0040,R0127,low_lexical_score,high,1,1.0,0.0
11,D0099,R0052,negation,high,1,1.0,0.0
12,D0088,R0219,materials,high,3,3.0,0.0
1,D0056,R0172,random,high,1,2.0,-1.0


In [70]:
both_present = (
    e6_e7_comparison["e6_rank"].notna()
    & e6_e7_comparison["e7_rank"].notna()
)

rank_change_summary = pd.Series(
    {
        "improved": int(
            (
                both_present
                & e6_e7_comparison[
                    "change_vs_e6"
                ].gt(0)
            ).sum()
        ),
        "unchanged": int(
            (
                both_present
                & e6_e7_comparison[
                    "change_vs_e6"
                ].eq(0)
            ).sum()
        ),
        "worsened": int(
            (
                both_present
                & e6_e7_comparison[
                    "change_vs_e6"
                ].lt(0)
            ).sum()
        ),
        "recovered_in_top_10": int(
            (
                e6_e7_comparison["e6_rank"].isna()
                & e6_e7_comparison[
                    "e7_rank"
                ].notna()
            ).sum()
        ),
        "lost_from_top_10": int(
            (
                e6_e7_comparison[
                    "e6_rank"
                ].notna()
                & e6_e7_comparison[
                    "e7_rank"
                ].isna()
            ).sum()
        ),
    },
    name="query_count",
)

display(rank_change_summary)

improved               1
unchanged              8
worsened               5
recovered_in_top_10    0
lost_from_top_10       1
Name: query_count, dtype: int64

### Результат E7

- E7 получил `Hit@1 = 0,400`, `Hit@5 = 0,933`,
  `Hit@10 = 0,933` и `MRR@10 = 0,602`, уступив E6
  по трем из четырех метрик

- Относительно E6 ранг ожидаемого регламента улучшился для
  одного запроса, не изменился для восьми и ухудшился для пяти.
  Регламент `R0268` для `D0085` выпал из top-10

- При финальном прогоне загрузка модели из локального кеша
  заняла около 1,5 секунды. Переранжирование 3600 пар
  на CPU заняло около 361 секунды

E7 не включается в финальную конфигурацию. Дополнительные модели
и параметры не проверяются, чтобы не подбирать решение под
15 dev-примеров

## Финальная high-confidence проверка

In [71]:
e6_e7_high_confidence = pd.DataFrame(
    [
        {
            "experiment": "E6",
            **calculate_ranking_metrics(
                e6_predictions,
                high_confidence_dev,
            ),
        },
        {
            "experiment": "E7",
            **calculate_ranking_metrics(
                e7_predictions,
                high_confidence_dev,
            ),
        },
    ]
)

display(e6_e7_high_confidence.round(3))

,experiment,n_queries,hit_at_1,hit_at_5,hit_at_10,mrr_at_10
0,E6,13,0.615,1.0,1.0,0.795
1,E7,13,0.462,1.0,1.0,0.669


### Вывод по high-confidence части

- На 13 примерах с высокой уверенностью E6 получил
`Hit@1 = 0,615`, `Hit@5 = 1,000`, `Hit@10 = 1,000`
и `MRR@10 = 0,795`

- На этой же выборке E7 сохранил `Hit@10 = 1,000`,
  но снизил `Hit@1` до 0,462 и `MRR@10` до 0,669.
  Следовательно, проигрыш E7 наблюдается и без двух меток
  со средней уверенностью не объясняется только двумя примерами со средней уверенностью

- E6 остается лучшей конфигурацией и на полной dev-выборке,
и на ее high-confidence части

## Финальный выбор конфигурации

- По результатам dev-экспериментов выбрана E6 - Reciprocal Rank Fusion
полных ранжирований `char_wb TF-IDF` и `multilingual-e5-small`

- E6 получила максимальный `MRR@10 = 0,722` и нашла ожидаемый
регламент в top-10 для всех 15 dev-примеров. В отличие от E2,
она также вернула `R0268` для декларации `D0085`

- Конфигурация и параметры E6 фиксируются до просмотра holdout
и после его оценки не изменяются

In [72]:
holdout_labels = manual_evaluation.query(
    "split == 'holdout'"
).copy()

assert len(holdout_labels) == 5

e6_holdout_metrics = calculate_ranking_metrics(
    e6_predictions,
    holdout_labels,
    ks=(1, 5, 10),
)

e6_holdout_ranks = get_expected_ranks(
    e6_predictions,
    holdout_labels,
)

holdout_analysis = holdout_labels[
    [
        "declaration_id",
        "expected_regulation_id",
        "confidence",
    ]
].copy()

holdout_analysis["e6_rank"] = (
    holdout_analysis["declaration_id"]
    .map(e6_holdout_ranks)
)

display(pd.Series(e6_holdout_metrics))
display(holdout_analysis)

n_queries    5.0
hit_at_1     0.0
hit_at_5     0.6
hit_at_10    0.6
mrr_at_10    0.2
dtype: float64

,declaration_id,expected_regulation_id,confidence,e6_rank
15,D0048,R0243,high,3.0
16,D0008,R0275,high,3.0
17,D0057,R0225,high,3.0
18,D0046,R0315,medium,NaN
19,D0014,R0238,high,NaN


### Результат holdout

- Финальная конфигурация E6 была один раз проверена на пяти
holdout-примерах после завершения выбора модели

- E6 получила `Hit@1 = 0,000`, `Hit@5 = 0,600`,
`Hit@10 = 0,600` и `MRR@10 = 0,200`. Для трех деклараций
ожидаемый регламент оказался на третьем месте, а для `D0046`
и `D0014` не попал в top-10

- Результат ниже dev-метрик. Однако при `n=5` один запрос меняет
`Hit@k` на 0,2, поэтому holdout используется как заключительная
проверка, а не как надежная оценка качества на новых данных

После просмотра результата конфигурация E6 не изменялась

In [73]:
holdout_candidates = (
    e6_predictions[
        e6_predictions["declaration_id"].isin(
            holdout_labels["declaration_id"]
        )
    ]
    .merge(
        regulations[
            [
                "regulation_id",
                "code",
                "description",
            ]
        ],
        on="regulation_id",
        how="left",
    )
    .merge(
        holdout_labels[
            [
                "declaration_id",
                "expected_regulation_id",
            ]
        ],
        on="declaration_id",
        how="left",
    )
)

holdout_candidates["is_expected"] = (
    holdout_candidates["regulation_id"]
    .eq(
        holdout_candidates[
            "expected_regulation_id"
        ]
    )
)

display(
    holdout_candidates.loc[
        holdout_candidates["rank"].le(3),
        [
            "declaration_id",
            "rank",
            "regulation_id",
            "code",
            "description",
            "is_expected",
        ],
    ]
)

,declaration_id,rank,regulation_id,code,description,is_expected
0,D0008,1,R0276,8483109500,Прочие валы трансмиссионные (включая кулачковы...,False
1,D0008,2,R0253,8413302001,"насосы топливные, для промышленной сборки мото...",False
2,D0008,3,R0275,8483105000,Шарнирные валы,True
10,D0014,1,R0127,3909509001,Прочие полиуретаны для производства волокон оп...,False
11,D0014,2,R0128,3909509002,Прочие полиуретаны для кожевенно-обувной промы...,False
12,D0014,3,R0219,7223001909,"Проволока из коррозионностойкой стали, содержа...",False
20,D0046,1,R0316,8703606021,Автомобили легковые и прочие моторные транспор...,False
21,D0046,2,R0319,8703606024,Автомобили легковые и прочие моторные транспор...,False
22,D0046,3,R0242,8407219100,Двигатели внутреннего сгорания с искровым зажи...,False
30,D0048,1,R0242,8407219100,Двигатели внутреннего сгорания с искровым зажи...,False


### Качественный разбор holdout

- В трех найденных случаях ожидаемый регламент занимает третье
  место. Основная ошибка этих примеров связана с порядком
  кандидатов внутри найденной товарной категории

- Для D0046 модель нашла несколько соседних автомобильных
регламентов, но не включила ожидаемый R0315 в top-10.
Метка этого примера имеет среднюю уверенность

- Для D0014 произошла ошибка поиска кандидата: модель отдала
приоритет регламентам, связанным с материалом изделия,
и не восстановила регламент для поворотного колеса офисной
мебели. Этот пример показывает ограничение общего текстового
сходства при различении типа изделия и материала

- Разбор фиксирует ограничения выбранного подхода и не используется
  для изменения модели

## Итог

По результатам dev-экспериментов выбрана E6 — объединение полного
ранжирования `char_wb TF-IDF` и `multilingual-e5-small`
с помощью Reciprocal Rank Fusion

Основные результаты:

- char TF-IDF превзошел word TF-IDF и BM25
- расширение документов дополнительными полями и иерархией
  не дало стабильного улучшения
- E5 не превзошла E2 самостоятельно, но вернула кандидата,
  отсутствовавшего в top-10 E2
- E6 показала лучший `MRR@10` на dev
- переранжирование cross-encoder-моделью ухудшило качество
  и значительно увеличило время выполнения

| Выборка | Запросов | Hit@1 | Hit@5 | Hit@10 | MRR@10 |
|---|---:|---:|---:|---:|---:|
| Dev | 15 | 0,533 | 0,933 | 1,000 | 0,722 |
| Holdout | 5 | 0,000 | 0,600 | 0,600 | 0,200 |

Holdout был проверен один раз после фиксации E6. Результат ниже
dev-метрик, но размер обеих ручных выборок слишком мал для устойчивой
оценки качества

Поля `G011` и `G34` не использовались, поскольку в регламентах
нет сопоставимых структурированных признаков направления перемещения
и страны. Поля разрешительных документов исключены из-за риска
процессной утечки

Финальная конфигурация переносится в `run.py` без использования
ручной разметки